# Agrupamento de variedades de arroz

A base de dados é composta por **45** características e **2'091** entradas.
Removendo duplicatas, restam **1'812** entradas.


In [ ]:
%%html
<link rel="stylesheet" href="./style.css">

In [ ]:
import import_ipynb  # noqa: F401
from treatment import (  # type: ignore
    f3_categorical_columns,
    f3_df,
    f3_numerical_columns,
)


In [ ]:
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from kmodes.kmodes import KModes
from kmodes.kprototypes import KPrototypes
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from columns import target_columns
from data import a_df
from plotting import (
    plot_boxplot_pair_with_clusters,
    plot_boxplots_with_clusters_for_numerical_columns,
    plot_cluster_as_pillar_graph,
    plot_numerical_by_categorical_and_cluster,
    plot_scatterplot_pair_with_clusters,
)

In [ ]:
def visualize_clustering(data_frame: pd.DataFrame):
    plot_boxplots_with_clusters_for_numerical_columns(
        data_frame=data_frame,
        numerical_columns=[*f3_numerical_columns, *target_columns],
    )

    plot_cluster_as_pillar_graph(data_frame=data_frame, categorical_column="agriblock")
    plot_cluster_as_pillar_graph(data_frame=data_frame, categorical_column="variety")
    plot_cluster_as_pillar_graph(data_frame=data_frame, categorical_column="soil_type")
    plot_cluster_as_pillar_graph(data_frame=data_frame, categorical_column="nursery")

    for x_axis, y_axis in combinations([*f3_numerical_columns, *target_columns], 2):
        plot_scatterplot_pair_with_clusters(
            data_frame=data_frame,
            x_axis=x_axis,
            y_axis=y_axis,
        )

    plot_boxplot_pair_with_clusters(
        data_frame=data_frame, x_axis="agriblock", y_axis="paddy_yield_per_hectare"
    )
    plot_boxplot_pair_with_clusters(
        data_frame=data_frame, x_axis="agriblock", y_axis="trash_per_hectare"
    )
    plot_boxplot_pair_with_clusters(
        data_frame=data_frame, x_axis="agriblock", y_axis="hectares"
    )

    plot_boxplot_pair_with_clusters(
        data_frame=data_frame, x_axis="variety", y_axis="paddy_yield_per_hectare"
    )
    plot_boxplot_pair_with_clusters(
        data_frame=data_frame, x_axis="variety", y_axis="trash_per_hectare"
    )
    plot_boxplot_pair_with_clusters(
        data_frame=data_frame, x_axis="variety", y_axis="hectares"
    )

    plot_boxplot_pair_with_clusters(
        data_frame=data_frame, x_axis="soil_type", y_axis="paddy_yield_per_hectare"
    )
    plot_boxplot_pair_with_clusters(
        data_frame=data_frame, x_axis="soil_type", y_axis="trash_per_hectare"
    )
    plot_boxplot_pair_with_clusters(
        data_frame=data_frame, x_axis="soil_type", y_axis="hectares"
    )

    plot_boxplot_pair_with_clusters(
        data_frame=data_frame, x_axis="nursery", y_axis="paddy_yield_per_hectare"
    )
    plot_boxplot_pair_with_clusters(
        data_frame=data_frame, x_axis="nursery", y_axis="trash_per_hectare"
    )
    plot_boxplot_pair_with_clusters(
        data_frame=data_frame, x_axis="nursery", y_axis="hectares"
    )

    plot_numerical_by_categorical_and_cluster(
        data_frame=data_frame,
        numerical_column="paddy_yield_per_hectare",
        categorical_column="variety",
    )
    plot_numerical_by_categorical_and_cluster(
        data_frame=data_frame,
        numerical_column="trash_per_hectare",
        categorical_column="variety",
    )
    plot_numerical_by_categorical_and_cluster(
        data_frame=data_frame,
        numerical_column="paddy_yield_per_hectare",
        categorical_column="agriblock",
    )
    plot_numerical_by_categorical_and_cluster(
        data_frame=data_frame,
        numerical_column="trash_per_hectare",
        categorical_column="agriblock",
    )
    plot_numerical_by_categorical_and_cluster(
        data_frame=data_frame,
        numerical_column="paddy_yield_per_hectare",
        categorical_column="soil_type",
    )
    plot_numerical_by_categorical_and_cluster(
        data_frame=data_frame,
        numerical_column="trash_per_hectare",
        categorical_column="soil_type",
    )
    plot_numerical_by_categorical_and_cluster(
        data_frame=data_frame,
        numerical_column="paddy_yield_per_hectare",
        categorical_column="nursery",
    )
    plot_numerical_by_categorical_and_cluster(
        data_frame=data_frame,
        numerical_column="trash_per_hectare",
        categorical_column="nursery",
    )
    plot_numerical_by_categorical_and_cluster(
        data_frame=data_frame,
        numerical_column="paddy_yield_per_hectare",
        categorical_column="hectares",
    )
    plot_numerical_by_categorical_and_cluster(
        data_frame=data_frame,
        numerical_column="trash_per_hectare",
        categorical_column="hectares",
    )

## K-Prototypes


In [ ]:
def cluster_by_k_prototypes(n_clusters: int):
    features = f3_categorical_columns + f3_numerical_columns
    clustered_df_by_kp = f3_df[features].copy()

    # K-Prototypes expects categorical columns as strings
    for col in f3_categorical_columns:
        clustered_df_by_kp[col] = clustered_df_by_kp[col].astype(str)

    # Identify indexes of categorical indexes
    indexes_of_categorical_columns = [
        clustered_df_by_kp.columns.get_loc(col) for col in f3_categorical_columns
    ]

    # Set parameters
    k_prototypes_protocol = KPrototypes(n_clusters=n_clusters, random_state=42)

    # Clustering
    clusters = k_prototypes_protocol.fit_predict(
        clustered_df_by_kp.to_numpy(), categorical=indexes_of_categorical_columns
    )

    # Add column describing the cluster to the database
    clustered_df_by_kp["cluster"] = clusters

    # Add target variables back for cluster analysis
    clustered_df_by_kp["paddy_yield_per_hectare"] = f3_df["paddy_yield_per_hectare"]
    clustered_df_by_kp["trash_per_hectare"] = f3_df["trash_per_hectare"]

    # Revert scaling
    clustered_df_by_kp["hectares"] = a_df["hectares"]
    clustered_df_by_kp["paddy_yield_per_hectare"] = a_df["paddy_yield_per_hectare"]
    clustered_df_by_kp["trash_per_hectare"] = a_df["trash_per_hectare"]

    return clustered_df_by_kp, k_prototypes_protocol.cost_


In [ ]:
def evaluate_k_prototypes(data_frame, categorical_columns, numerical_columns, cost):
    # Labels dos clusters
    labels = data_frame["cluster"].to_numpy()

    # Número de clusters
    k = data_frame["cluster"].nunique()

    # Tamanho dos clusters
    cluster_sizes = data_frame["cluster"].value_counts()

    # Dados categóricos
    X_categorical = data_frame[categorical_columns].astype(str).to_numpy()

    # Dados numéricos
    X_numerical = data_frame[numerical_columns].to_numpy()

    # Padronizar variáveis numéricas
    scaler = StandardScaler()
    X_numerical = scaler.fit_transform(X_numerical)

    # Distância categórica (matching dissimilarity)
    categorical_distance = np.mean(
        X_categorical[:, np.newaxis, :] != X_categorical[np.newaxis, :, :], axis=2
    )

    # Distância euclidiana para variáveis numéricas
    numerical_distance = np.sqrt(
        np.sum(
            (X_numerical[:, np.newaxis, :] - X_numerical[np.newaxis, :, :]) ** 2, axis=2
        )
    )

    # Normalizar a distância numérica para ficar
    # aproximadamente na mesma escala da categórica
    numerical_distance = numerical_distance / numerical_distance.max()

    # Combinar as distâncias
    distance_matrix = (categorical_distance + numerical_distance) / 2

    # Silhouette
    silhouette = silhouette_score(distance_matrix, labels, metric="precomputed")

    # Resultado
    return pd.DataFrame(
        {
            "K": [k],
            "Custo": [cost],
            "Silhouette": [silhouette],
            "Menor cluster": [cluster_sizes.min()],
            "Maior cluster": [cluster_sizes.max()],
        }
    )

In [ ]:
def evaluate_multiple_k_prototypes(
    min_k, max_k, categorical_columns, numerical_columns
):
    evaluations = []

    # Avaliar cada K
    for k in range(min_k, max_k + 1):
        clustering, cost = cluster_by_k_prototypes(n_clusters=k)

        evaluation = evaluate_k_prototypes(
            data_frame=clustering,
            categorical_columns=categorical_columns,
            numerical_columns=numerical_columns,
            cost=cost,
        )

        evaluations.append(evaluation)

    # Consolidar resultados
    evaluation_df = pd.concat(evaluations, ignore_index=True)

    # Exibir tabela completa
    display(evaluation_df)

    # Gráfico do custo
    plt.figure(figsize=(8, 5))

    plt.plot(evaluation_df["K"], evaluation_df["Custo"], marker="o")

    plt.xlabel("Número de clusters (K)")
    plt.ylabel("Custo")
    plt.title("Custo do K-Prototypes por número de clusters")
    plt.xticks(evaluation_df["K"])
    plt.grid(True)
    plt.show()

    # Gráfico da Silhouette
    plt.figure(figsize=(8, 5))

    plt.plot(evaluation_df["K"], evaluation_df["Silhouette"], marker="o")

    plt.xlabel("Número de clusters (K)")
    plt.ylabel("Silhouette")
    plt.title("Silhouette por número de clusters")
    plt.xticks(evaluation_df["K"])
    plt.grid(True)
    plt.show()

    return evaluation_df

In [ ]:
evaluation_df = evaluate_multiple_k_prototypes(
    min_k=2,
    max_k=8,
    categorical_columns=f3_categorical_columns,
    numerical_columns=f3_numerical_columns,
)

In [ ]:
clustering, _ = cluster_by_k_prototypes(n_clusters=2)
visualize_clustering(data_frame=clustering)


## K-Modes


In [ ]:
def cluster_by_k_modes(n_clusters: int):
    clustered_df_by_km = f3_df[f3_categorical_columns].copy()

    # K-Modes expects categorical columns as strings
    for col in f3_categorical_columns:
        clustered_df_by_km[col] = clustered_df_by_km[col].astype(str)

    # Set parameters
    k_modes_model = KModes(n_clusters=n_clusters, random_state=42)

    # Clustering
    clusters = k_modes_model.fit_predict(clustered_df_by_km.to_numpy())

    # Add column describing the cluster to the database
    clustered_df_by_km["cluster"] = clusters

    # Add target variables back for cluster analysis
    clustered_df_by_km["hectares"] = f3_df["hectares"]
    clustered_df_by_km["paddy_yield_per_hectare"] = f3_df["paddy_yield_per_hectare"]
    clustered_df_by_km["trash_per_hectare"] = f3_df["trash_per_hectare"]

    # Revert scaling
    clustered_df_by_km["hectares"] = a_df["hectares"]
    clustered_df_by_km["paddy_yield_per_hectare"] = a_df["paddy_yield_per_hectare"]
    clustered_df_by_km["trash_per_hectare"] = a_df["trash_per_hectare"]

    return clustered_df_by_km, k_modes_model.cost_

In [ ]:
def evaluate_k_modes(data_frame, categorical_columns, cost):
    # Labels dos clusters
    labels = data_frame["cluster"].to_numpy()

    # Número de clusters
    k = data_frame["cluster"].nunique()

    # Tamanho dos clusters
    cluster_sizes = data_frame["cluster"].value_counts()

    # Dados categóricos
    X = data_frame[categorical_columns].astype(str).to_numpy()

    # Distância categórica (matching dissimilarity)
    distance_matrix = np.mean(X[:, np.newaxis, :] != X[np.newaxis, :, :], axis=2)

    # Silhouette para dados categóricos
    silhouette = silhouette_score(distance_matrix, labels, metric="precomputed")

    # Criar tabela de avaliação
    return pd.DataFrame(
        {
            "K": [k],
            "Custo": [cost],
            "Silhouette": [silhouette],
            "Menor cluster": [cluster_sizes.min()],
            "Maior cluster": [cluster_sizes.max()],
        }
    )


In [ ]:
def evaluate_multiple_k_modes(min_k, max_k, categorical_columns):
    evaluations = []

    # Avaliar cada K
    for k in range(min_k, max_k + 1):
        clustering, cost = cluster_by_k_modes(n_clusters=k)

        evaluation = evaluate_k_modes(
            data_frame=clustering, categorical_columns=categorical_columns, cost=cost
        )

        evaluations.append(evaluation)

    # Consolidar resultados
    evaluation_df = pd.concat(evaluations, ignore_index=True)

    # Exibir tabela completa
    display(evaluation_df)

    # Gráfico do custo
    plt.figure(figsize=(8, 5))

    plt.plot(evaluation_df["K"], evaluation_df["Custo"], marker="o")

    plt.xlabel("Número de clusters (K)")
    plt.ylabel("Custo")
    plt.title("Custo do K-Modes por número de clusters")
    plt.xticks(evaluation_df["K"])
    plt.grid(True)
    plt.show()

    # Gráfico da Silhouette
    plt.figure(figsize=(8, 5))

    plt.plot(evaluation_df["K"], evaluation_df["Silhouette"], marker="o")

    plt.xlabel("Número de clusters (K)")
    plt.ylabel("Silhouette")
    plt.title("Silhouette por número de clusters")
    plt.xticks(evaluation_df["K"])
    plt.grid(True)
    plt.show()

    return evaluation_df

In [ ]:
evaluation_df = evaluate_multiple_k_modes(
    min_k=2, max_k=8, categorical_columns=f3_categorical_columns
)

In [ ]:
clustering, _ = cluster_by_k_modes(n_clusters=2)
visualize_clustering(data_frame=clustering)